In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

model = init_chat_model("google_genai:gemini-2.5-flash")
response = model.invoke("Capital of India")
response.content

'New Delhi'

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in India?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "India"}'}, '__gemini_function_call_thought_signatures__': {'b7b0a2dd-385f-4ae2-ad48-24a22a56bdad': 'CogCAQw51sdxxmg0rb1qz8LyVB2KHJimO9dV56E1Eq2wULq1zn0KHq5/5QLTyiis1UpIHadKGnjZ9emNl7JNme8JJdqYxnUCcFwIrBQm1CcaUsl5WM4qgsxbmEWhVe0aUnHayoW3TUAKTfT7ycbiNvd41vmPiPdRkNsMWCSCmixRF8agODU8fltSWJ7hoRTG2SLguwf89mw6zh/9TmQ99fYxvsVcWo4lO7UyofEoTIHzVU+GkAxMOw2CHC/BJiwygGGTyi8h4GffnuBnaNhsFUKj0yRgxIxJyBdgFCGgu48U+ZXJRSMnlND+go2QhYUOhqcAeEereeRvDEvU1Bv5O2j11FAoVMxjM4m1'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f079f-a449-77c3-b086-eb9b5228d460-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'India'}, 'id': 'b7b0a2dd-385f-4ae2-ad48-24a22a56bdad', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 71, 'total_tokens': 119, 'input_token_details

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in India?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in India
